# Chapter 02 과제 — VS Code에서 시작하는 데이터 분석 환경

이 Notebook은 Chapter 02 실습 과제를 한 번에 점검할 수 있도록 작성한 제출용 코드입니다.

## 목표
- 현재 Notebook이 사용하는 Python 실행 파일과 작업 폴더 확인
- 프로젝트 루트와 `data/raw` 경로 자동 탐색
- 필수 패키지 import 및 버전 확인
- 샘플 CSV 4개 존재 여부 확인
- `customers.csv` 로드 및 기본 구조 확인
- `.env`가 Git 추적 대상이 아닌지 확인
- 마지막 셀에서 제출용 체크 결과 한 번에 확인

> VS Code에서 **프로젝트의 `.venv` 커널**을 선택한 뒤 `Run All` 하세요.


## 1. Python / Notebook 실행 환경 확인


In [ ]:
from pathlib import Path
import sys
import platform

print("Python 버전:", sys.version.split()[0])
print("Python 실행 파일:", sys.executable)
print("현재 작업 폴더:", Path.cwd())
print("운영체제:", platform.platform())

is_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
print("가상환경 사용 여부:", is_venv)
print("sys.prefix:", sys.prefix)


## 2. 프로젝트 루트와 데이터 경로 자동 탐색


In [ ]:
cwd = Path.cwd().resolve()

# Notebook을 프로젝트 루트에서 실행해도, notebooks 폴더에서 실행해도 동작하도록 탐색
candidates = [cwd, cwd.parent]

PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / "requirements.txt").exists() and (candidate / "data" / "raw").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. "
        "VS Code에서 -llm-data-analysis-course 폴더를 열고 다시 실행하세요."
    )

DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)
print("DATA_DIR.exists():", DATA_DIR.exists())


## 3. 필수 패키지 import 및 버전 확인


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)


## 4. 샘플 CSV 4개 존재 여부 확인


In [ ]:
required_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

file_status = {
    filename: (DATA_DIR / filename).exists()
    for filename in required_files
}

for filename, exists in file_status.items():
    print(f"{filename}: {'OK' if exists else 'MISSING'}")

missing_files = [name for name, exists in file_status.items() if not exists]

if missing_files:
    raise FileNotFoundError(
        "다음 파일이 없습니다: "
        + ", ".join(missing_files)
        + "\n프로젝트 루트 터미널에서 `python scripts/generate_sample_data.py`를 실행하세요."
    )


## 5. `customers.csv` 불러오기


In [ ]:
customers_path = DATA_DIR / "customers.csv"
customers = pd.read_csv(customers_path)

print("불러온 파일:", customers_path)
display(customers.head())


## 6. `customers` 데이터 기본 구조 확인


In [ ]:
print("customers.shape:", customers.shape)
print("컬럼명:", customers.columns.tolist())
print("\n--- customers.info() ---")
customers.info()


## 7. 기본 품질 점검


In [ ]:
print("결측값 개수")
display(customers.isna().sum().to_frame("missing_count"))

print("완전 중복 행 수:", customers.duplicated().sum())
print("customer_id 중복 수:", customers["customer_id"].duplicated().sum())


## 8. 나머지 CSV도 정상 로드되는지 확인


In [ ]:
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

summary = pd.DataFrame({
    "file": ["customers.csv", "products.csv", "orders.csv", "order_items.csv"],
    "rows": [len(customers), len(products), len(orders), len(order_items)],
    "columns": [
        customers.shape[1],
        products.shape[1],
        orders.shape[1],
        order_items.shape[1],
    ],
})

display(summary)


## 9. `.env` / Secret Git 추적 여부 확인

실제 Secret 값은 출력하지 않습니다.  
Git이 설치되어 있고 현재 저장소가 정상 clone된 상태라면 아래 셀이 `.env` 추적 여부를 확인합니다.


In [ ]:
import subprocess

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()

git_ok, git_root, git_err = run_git("rev-parse", "--show-toplevel")

if git_ok != 0:
    print("Git 저장소 확인 실패:", git_err)
    env_not_tracked = False
else:
    print("Git 저장소 루트:", git_root)

    _, tracked_env, _ = run_git("ls-files", ".env")
    env_not_tracked = tracked_env == ""

    print(".env Git 추적 여부:", "추적 안 됨 (정상)" if env_not_tracked else "추적 중 (확인 필요)")

    ignore_code, _, _ = run_git("check-ignore", "-q", ".env")
    print(".env .gitignore 적용 여부:", "적용됨" if ignore_code == 0 else "확인 필요")


## 10. 제출용 Evidence 값 한 번에 출력


In [ ]:
print("=" * 70)
print("Chapter 02 제출용 핵심 Evidence")
print("=" * 70)
print("1. Notebook sys.executable :", sys.executable)
print("2. Notebook Path.cwd()     :", Path.cwd())
print("3. PROJECT_ROOT            :", PROJECT_ROOT)
print("4. DATA_DIR                 :", DATA_DIR)
print("5. DATA_DIR.exists()        :", DATA_DIR.exists())
print("6. customers.shape          :", customers.shape)
print("7. customers.columns        :", customers.columns.tolist())
print("8. CSV 4개 존재            :", all(file_status.values()))
print("9. .env Git 추적 안 됨     :", env_not_tracked)
print("=" * 70)


## 11. 자동 최종 체크


In [ ]:
checks = {
    "가상환경(.venv 계열) 사용": is_venv,
    "DATA_DIR 존재": DATA_DIR.exists(),
    "CSV 4개 존재": all(file_status.values()),
    "customers 로드 성공": isinstance(customers, pd.DataFrame) and not customers.empty,
    "customers 컬럼 존재": len(customers.columns) > 0,
    ".env Git 추적 안 됨": env_not_tracked,
}

for item, passed in checks.items():
    print(f"[{'PASS' if passed else 'CHECK'}] {item}")

if all(checks.values()):
    print("\nChapter 02 코드 실행 점검: PASS")
else:
    print("\nCHECK 항목이 있습니다. 위 결과를 확인하세요.")


## 12. 오류가 생겼을 때 LLM에 질문할 안전한 템플릿

오류가 발생하면 아래 형식으로 질문합니다.  
**API Key, Token, 비밀번호, `.env` 실제 내용, 개인정보는 붙여넣지 않습니다.**


In [ ]:
safe_prompt = f"""
VS Code Jupyter Notebook에서 오류가 발생했습니다.

- 운영체제: {platform.system()}
- Python 실행 파일: {sys.executable}
- 현재 작업 폴더: {Path.cwd()}
- 프로젝트 루트: {PROJECT_ROOT}
- 데이터 폴더: {DATA_DIR}

실행한 코드:
[오류가 난 코드만 붙여넣기]

오류 메시지:
[Secret/개인정보를 제거한 오류 메시지 붙여넣기]

현재 작업 폴더와 Python 환경을 기준으로,
초보자가 이해할 수 있게 원인과 확인 순서를 설명해 주세요.
""".strip()

print(safe_prompt)
